In [1]:
import duckdb

conn = duckdb.connect()

conn.sql("""
CREATE OR REPLACE VIEW fec AS
SELECT
    JournalCode,
    JournalLib,
    EcritureNum,
    
    EcritureDate as EcritureDateRaw,
    strptime(
        CAST(EcritureDate as VARCHAR),
        '%Y%m%d'
    ) AS EcritureDate,
    
    CAST(CompteNum AS VARCHAR) AS CompteNum,
    CompteLib,
    CompAuxNum,
    CompAuxLib,
    PieceRef,
    PieceDate as PieceDateRaw,
    strptime(
        CAST(PieceDate as VARCHAR),
        '%Y%m%d'
    ) AS PieceDate,
    
    EcritureLib,
    EcritureLet,
    DateLet,
    ValidDate,
    --Montantdevise,
    --Idevise,
    CAST(
        replace(Debit, ',', '.') 
        AS DECIMAL(15,2)
    ) AS Debit,

    CAST(
        replace(Credit, ',', '.') 
        AS DECIMAL(15,2)
    ) AS Credit,
    
    CAST(
        replace(Credit, ',', '.') 
        AS DECIMAL(15,2)
    )  
    -
    CAST(
        replace(Debit, ',', '.') 
        AS DECIMAL(15,2)
    ) AS Solde
FROM read_csv(
    '/home/jovyan/data/FEC.txt',
    delim='\t',
    header=true
)
""")

In [2]:
conn.sql("""
SELECT
    JournalCode, 
    JournalLib,
    CompteNum,
    CompAuxNum,
    Debit, 
    Credit,
    Solde
FROM fec LIMIT 5
""")

┌─────────────┬────────────┬───────────┬────────────┬───────────────┬───────────────┬───────────────┐
│ JournalCode │ JournalLib │ CompteNum │ CompAuxNum │     Debit     │    Credit     │     Solde     │
│   varchar   │  varchar   │  varchar  │  varchar   │ decimal(15,2) │ decimal(15,2) │ decimal(16,2) │
├─────────────┼────────────┼───────────┼────────────┼───────────────┼───────────────┼───────────────┤
│ ac          │ Achats     │ 612214    │ NULL       │        708.85 │          0.00 │       -708.85 │
│ ac          │ Achats     │ 616110    │ NULL       │          9.11 │          0.00 │         -9.11 │
│ ac          │ Achats     │ 445661    │ NULL       │        141.77 │          0.00 │       -141.77 │
│ ac          │ Achats     │ 401000    │ FLIXXB     │          0.00 │        859.73 │        859.73 │
│ ac          │ Achats     │ 612210    │ NULL       │        545.89 │          0.00 │       -545.89 │
└─────────────┴────────────┴───────────┴────────────┴───────────────┴─────────────

In [3]:
import pandas as pd

def format_compta(value):
    if value == 0: 
        return ""
    return f"{value:,.2f}".replace(",", " ").replace(".", ",")

def style_total(row):
    if row["CompteNum"] == "":
        return ["font-weight: bold"] * len(row)
    return [""] * len(row)

df = conn.sql("""
SELECT
    CASE
        WHEN left(CompteNum,1)='1' THEN '1. Capitaux'
        WHEN left(CompteNum,1)='2' THEN '2. Immobilisations'
        WHEN left(CompteNum,1)='3' THEN '3. Stocks'
        WHEN left(CompteNum,1)='4' THEN '4. Tiers'
        WHEN left(CompteNum,1)='5' THEN '5. Financier'
        WHEN left(CompteNum,1)='6' THEN '6. Charges'
        WHEN left(CompteNum,1)='7' THEN '7. Produits'
    END AS Section,
        
    CompteNum,
    CompteLib,

    Count(*) As Nb,
    
    SUM(Debit) as Debit,
    SUM(Credit) as Credit,

    CASE
        WHEN SUM(Debit - Credit) > 0
        THEN SUM(Debit - Credit)
        ELSE 0
    END AS SoldeDebiteur,

    CASE
        WHEN SUM(Debit - Credit) < 0
        THEN ABS(SUM(Debit - Credit))
        ELSE 0
    END AS SoldeCrediteur
FROM fec
--WHERE EcritureDate >= '2022-04-01'
GROUP BY
    Section,
    CompteNum, 
    CompteLib 
ORDER BY CompteNum;
""").df()

lignes = []

for Section, groupe in df.groupby("Section"):

    lignes.append(groupe)

    total = pd.DataFrame([{
        "Section": Section,
        "CompteNum": "",
        "CompteLib": f"Total classe {Section}",
        "Nb": groupe["Nb"].sum(),
        "Debit": groupe["Debit"].sum(),
        "Credit": groupe["Credit"].sum(),
        "SoldeDebiteur": groupe["SoldeDebiteur"].sum(),
        "SoldeCrediteur": groupe["SoldeCrediteur"].sum()
    }])

    lignes.append(total)

balance = pd.concat(lignes, ignore_index=True)

total_general = pd.DataFrame([{
    "Section": "",
    "CompteNum": "",
    "CompteLib": "TOTAL GENERAL",
    "Nb": df["Nb"].sum(),
    "Debit": df["Debit"].sum(),
    "Credit": df["Credit"].sum(),
    "SoldeDebiteur": df["SoldeDebiteur"].sum(),
    "SoldeCrediteur": df["SoldeCrediteur"].sum()
}])

balance = pd.concat(
    [balance, total_general],
    ignore_index=True
)

balance.style \
    .format({
        "Debit": format_compta,
        "Credit": format_compta,
        "SoldeDebiteur": format_compta,
        "SoldeCrediteur": format_compta
    }) \
    .set_properties(
        subset=["Section", "CompteNum", "CompteLib"],
        **{"text-align": "left"}
    ) \
    .apply(style_total, axis=1)

,Section,CompteNum,CompteLib,Nb,Debit,Credit,SoldeDebiteur,SoldeCrediteur
0,1. Capitaux,101300,CAPITAL SOUSCRIT APPELE VERSE,1,,"20 000,00",,"20 000,00"
1,1. Capitaux,106100,RESERVE LEGALE,1,,"2 000,00",,"2 000,00"
2,1. Capitaux,106800,AUTRES RESERVES,1,,"189 030,53",,"189 030,53"
3,1. Capitaux,110000,REPORT A NOUVEAU SOLDE CREDITEUR,2,,"1 001 453,99",,"1 001 453,99"
4,1. Capitaux,120000,RESULTAT DE L'EXERCICE BENEFICE,4,"777 977,75","777 977,75",,
5,1. Capitaux,151200,PROV.GARANT.AUX CLIENTS,2,,"170 000,00",,"170 000,00"
6,1. Capitaux,164000,DAILLY BECM 465211,4,"1 000 000,00","1 000 000,00",,
7,1. Capitaux,164100,PRET CAVF BAT 10000340040 42 000,13,"5 987,28","25 779,75",,"19 792,47"
8,1. Capitaux,164300,PRET CAVF PGE,3,"77 369,60","280 000,00",,"202 630,40"
9,1. Capitaux,164400,PRET BPI CROISSANCE,1,,"633 820,00",,"633 820,00"


In [4]:
conn.sql("""
SELECT
    JournalCode, 
    CompteNum,
    EcritureDate,
    PieceDate,
    Debit, 
    Credit
FROM fec 
WHERE CompteNum = '120000'
""")

┌─────────────┬───────────┬─────────────────────┬─────────────────────┬───────────────┬───────────────┐
│ JournalCode │ CompteNum │    EcritureDate     │      PieceDate      │     Debit     │    Credit     │
│   varchar   │  varchar  │      timestamp      │      timestamp      │ decimal(15,2) │ decimal(15,2) │
├─────────────┼───────────┼─────────────────────┼─────────────────────┼───────────────┼───────────────┤
│ AD          │ 120000    │ 2021-04-01 00:00:00 │ 2022-04-01 00:00:00 │          0.00 │     230819.79 │
│ AD          │ 120000    │ 2021-09-30 00:00:00 │ 2022-04-01 00:00:00 │     230819.79 │          0.00 │
│ AD          │ 120000    │ 2022-04-01 00:00:00 │ 2022-04-01 00:00:00 │          0.00 │     547157.96 │
│ OD          │ 120000    │ 2022-08-24 00:00:00 │ 2022-08-24 00:00:00 │     547157.96 │          0.00 │
└─────────────┴───────────┴─────────────────────┴─────────────────────┴───────────────┴───────────────┘

In [5]:
import pandas as pd
pd.set_option("display.max_rows", None)

df = conn.sql("""
SELECT
    JournalCode, 
    CompteNum,
    Year(EcritureDate) as Annee,
    Month(EcritureDate) as Mois,
    Sum(Debit) AS Debit, 
    Sum(Credit) AS Credit
FROM fec 
WHERE CompteNum = '401000' 
--AND EcritureDate < '2022-04-01' 
--AND Debit > 0 AND Debit < 4298
GROUP BY
    JournalCode,
    CompteNum, 
    Annee, 
    Mois
ORDER BY JournalCode, Annee, Mois
""").df()

lignes = []

for JournalCode, groupe in df.groupby("JournalCode"):
    lignes.append(groupe)

    total = pd.DataFrame([{
        "JournalCode": f"Total {JournalCode}",
        "CompteNum": "",
        "Annee": "",
        "Mois": "",
        "Debit": groupe["Debit"].sum(),
        "Credit": groupe["Credit"].sum()
    }])

    lignes.append(total)

balance = pd.concat(lignes, ignore_index=True)

total_general = pd.DataFrame([{
    "JournalCode": "TOTAL GENERAL",
    "CompteNum": "",
    "Annee": "",
    "Mois": "",
    "Debit": df["Debit"].sum(),
    "Credit": df["Credit"].sum()
}])

balance = pd.concat(
    [balance, total_general],
    ignore_index=True
)

balance.style \
    .format({
        "Debit": format_compta,
        "Credit": format_compta
    }) \
    .set_properties(
        subset=["JournalCode"],
        **{"text-align": "left"}
    ) \
    .apply(style_total, axis=1)

,JournalCode,CompteNum,Annee,Mois,Debit,Credit
0,ACB,401000,2023,3,"7 362,46","27 728,14"
1,Total ACB,,,,"7 362,46","27 728,14"
2,AD,401000,2020,5,"12,60","61,50"
3,AD,401000,2020,11,,"275,26"
4,AD,401000,2021,1,"1 800,00","6 000,00"
5,AD,401000,2021,2,,"2 994,01"
6,AD,401000,2021,3,,"688,43"
7,AD,401000,2021,4,"315,72","990,55"
8,AD,401000,2021,5,"1 903,17","163 612,39"
9,AD,401000,2021,6,"656,48","1 216,66"


In [70]:
df = conn.sql("""
SELECT
    JournalCode, 
    CompteNum,
    Year(EcritureDate) as Annee,
    Month(EcritureDate) as Mois,
    EcritureLib,
    Debit, 
    Credit
FROM fec 
WHERE CompteNum = '467120' 
--AND JournalCode = 'AD'
--AND EcritureDate < '2022-04-01' 
AND Debit > 0 
--AND Debit < 7200
ORDER BY JournalCode, Annee, Mois
""").df()

total_general = pd.DataFrame([{
    "JournalCode": "TOTAL GENERAL",
    "CompteNum": "",
    "Annee": "",
    "Mois": "",
    "Debit": df["Debit"].sum(),
    "Credit": df["Credit"].sum()
}])

balance = pd.concat(
    [df, total_general],
    ignore_index=True
)

balance.style \
    .format({
        "Debit": format_compta,
        "Credit": format_compta
    }) \
    .set_properties(
        subset=["JournalCode", "EcritureLib"],
        **{"text-align": "left"}
    ) \
    .apply(style_total, axis=1)

,JournalCode,CompteNum,Annee,Mois,EcritureLib,Debit,Credit
0,AD,467120,2021,6,ELABOR - PRIMES CEE LEFEBVRE,"80 000,00",
1,AD,467120,2021,9,TERRYLOIRE-PRIMES CEE,"4 056 559,22",
2,AD,467120,2022,1,ELABOR - PRIMES,"1 153 040,00",
3,AD,467120,2022,3,EARL LA BREBIDORE,"101 912,26",
4,OD,467120,2022,10,Ecart de règlement Primes CEE Réfrigération,"0,07",
5,OD,467120,2022,10,Ecart de règlement Primes CEE Réfrigération,"0,62",
6,OD,467120,2022,10,Ecart de règlement Primes CEE Réfrigération,"0,01",
7,VT1,467120,2022,6,EI GRANGE ROUGE CEE,"156 200,00",
8,VT1,467120,2022,10,ROY FLORIAN CEE DU 14.06.22,"80 636,00",
9,VT1,467120,2022,10,BENJAMIN DAUPHIN,"886 995,90",


In [73]:
df = conn.sql("""
SELECT
    JournalCode, 
    CompteNum,
    EcritureNum,
    Year(EcritureDate) as Annee,
    Month(EcritureDate) as Mois,
    EcritureLib,
    Debit, 
    Credit
FROM fec 
WHERE CompteNum = '467120' 
--AND JournalCode = 'AD'
--AND EcritureNum = 25
--AND EcritureDate < '2022-04-01' 
AND Debit > 0
ORDER BY JournalCode, Annee, Mois
""").df()

total_general = pd.DataFrame([{
    "JournalCode": "TOTAL GENERAL",
    "CompteNum": "",
    "Annee": "",
    "Mois": "",
    "Debit": df["Debit"].sum(),
    "Credit": df["Credit"].sum()
}])

balance = pd.concat(
    [df, total_general],
    ignore_index=True
)

balance.style \
    .format({
        "Debit": format_compta,
        "Credit": format_compta
    }) \
    .set_properties(
        subset=["JournalCode", "EcritureLib"],
        **{"text-align": "left"}
    ) \
    .apply(style_total, axis=1)

,JournalCode,CompteNum,EcritureNum,Annee,Mois,EcritureLib,Debit,Credit
0,AD,467120,25.000000,2021,6,ELABOR - PRIMES CEE LEFEBVRE,"80 000,00",
1,AD,467120,25.000000,2021,9,TERRYLOIRE-PRIMES CEE,"4 056 559,22",
2,AD,467120,25.000000,2022,1,ELABOR - PRIMES,"1 153 040,00",
3,AD,467120,25.000000,2022,3,EARL LA BREBIDORE,"101 912,26",
4,OD,467120,7759.000000,2022,10,Ecart de règlement Primes CEE Réfrigération,"0,07",
5,OD,467120,7762.000000,2022,10,Ecart de règlement Primes CEE Réfrigération,"0,62",
6,OD,467120,7764.000000,2022,10,Ecart de règlement Primes CEE Réfrigération,"0,01",
7,VT1,467120,5594.000000,2022,6,EI GRANGE ROUGE CEE,"156 200,00",
8,VT1,467120,5687.000000,2022,10,ROY FLORIAN CEE DU 14.06.22,"80 636,00",
9,VT1,467120,5696.000000,2022,10,BENJAMIN DAUPHIN,"886 995,90",
